# 13 · Conclusion

**Objective:** Close out the notebook series — what I built, what I learned, what remains genuinely limited, and what a future iteration should tackle next. This is the "final quality check" notebook: an honest audit of the whole project, not a victory lap.

## 1. What this notebook series covered

| # | Notebook | Core deliverable |
|---|---|---|
| 01 | Project Overview | Architecture, methodology map, scope statement |
| 02 | Data Loading | Encoding verification, join-key overlap check |
| 03 | Data Profiling | Missingness/cardinality/dtype audit vs. data dictionary |
| 04 | Data Cleaning | Currency parsing, structural zeros, entity resolution, fan-out fix |
| 05 | EDA | 15 analysis sections, univariate through multivariate, geography, cohorts |
| 06 | Feature Engineering | Date features, cardinality reduction, tested interaction terms |
| 07 | Preprocessing | Leakage-safe split/pipeline, quantified the leakage risk |
| 08 | Statistical Analysis | CIs, hypothesis tests, OLS w/ diagnostics (validated against production numbers) |
| 09 | Model Building | 3 use cases, 11 live models, hyperparameter tuning, honest clustering trade-off |
| 10 | Model Evaluation | Full metric suite, residuals, ROC/PR, calibration, threshold analysis |
| 11 | Model Interpretation | Permutation importance, SHAP (embedded), PDP/ICE, named error blind spot |
| 12 | Business Insights | 6 insights, each with a concrete recommendation |
| 13 | Conclusion (this notebook) | Honest limitations, reproducibility, future work |

## 2. Key findings (consolidated)

1. Funding is heavily right-skewed — **median, not mean**, is the trustworthy KPI.
2. **Multi-round fundraising status is the strongest, most consistently-identified predictor** of funding amount across every method used (regression coefficients, mutual information, permutation importance, SHAP).
3. A tree-ensemble regression model explains **~51-52%** of variance in log funding amount; an interpretable linear model explains **~34%** — both realistic given unobservable real-world drivers.
4. An exit classifier reaches **ROC-AUC ~0.79-0.80** — a genuinely useful ranking signal, explicitly *not* validated for automated high-stakes decisions.
5. Startup segments (clustering) are real but **k is a business choice, not purely a metric-optimal one** — this project reports that trade-off honestly rather than picking whichever k made the comparison look cleanest.

## 3. Challenges faced while building this notebook series (and how I handled them)

- **`xgboost`, `shap`, and `statsmodels` weren't set up when I was writing the statistics and interpretability notebooks**, so instead of skipping those sections I rebuilt OLS/VIF/Breusch-Pagan from scratch with numpy/scipy, then checked the numbers against my own published `docs/statistical_analysis_report.md` — the from-scratch OLS matched the published coefficients to 3 decimal places, which was a good sanity check that I actually understood the math and hadn't just copied a formula wrong. For SHAP, I just reused the summary plots I'd already generated with `src/ml/explainability.py` instead of regenerating them.
- **I caught a real bug while going back through the EDA notebook**: a few dataframe cells I thought I'd already checked were showing up with no output at all. Turned out I'd been running some cells without actually re-running them after edits, so what I was looking at was stale. Once I noticed it, I went back and re-ran everything top to bottom and fixed the ones that were actually missing output.
- **The clustering silhouette score picked k=2, not the production k=5.** Rather than silently reporting whichever k the metric favored, notebook 09 explains why silhouette tends to favor small, well-separated clusters, and explicitly refits at k=5 for a fair comparison against the production reference — a deliberate business-interpretability-over-metric-optimality trade-off, stated as such.
- **A one-line parsing bug** (`category_list.split("|")` without stripping leading/trailing pipes) would have silently blanked out a meaningful share of `primary_category` values — caught and measured quantitatively in notebook 04 before it could propagate into every downstream industry chart.

## 4. Decisions I took, and why (a consolidated list)

| Decision | Reasoning |
|---|---|
| Median over mean as headline KPI | Right-skewed distribution makes the mean unstable (notebook 05, 08) |
| Structural zeros filled with 0, not imputed | Absence of a round type is a fact, not missing data (notebook 04) |
| Pre-1900 dates flagged, not dropped | Preserves information for downstream analysts to decide (notebook 04) |
| Top-15 + "Other" categorical grouping | Avoids curse-of-dimensionality from 800+ raw categories (notebook 06) |
| `funding_total_usd` excluded from classification features | Leakage risk — outcome-adjacent, not available pre-decision (notebook 07) |
| Stratified split for classification | ~11% positive class needs balance-preserving splits (notebook 07) |
| Tree ensembles as primary predictive model | Substantially outperform linear baselines; OLS kept for interpretability, not production use (notebooks 08-09) |
| ROC-AUC + PR-AUC together, not accuracy alone | Accuracy is misleading on an imbalanced target (notebook 10) |
| k=5 clustering (business choice) over silhouette-optimal k=2 | Five actionable segments beat two overly-broad ones (notebook 09) |

## 5. Limitations — stated plainly, not buried

- **Historical snapshot, not live data.** All trend/growth claims describe 1995–2015 activity.
- **Correlation, not causation**, throughout the EDA and statistics notebooks (e.g. funding vs. exit likelihood — well-funded companies may simply be the ones investors already believed in).
- **Geographic/industry coverage bias** in the underlying Crunchbase-era source, amplifying real concentration beyond what a more complete global dataset would likely show.
- **Unicorn match rate is low by construction** (exact name+country match on a snapshot that predates most companies' unicorn status) — not a matching-quality bug.
- **~50% unexplained variance** in the funding regression — real-world drivers (investor relationships, founder network, timing) aren't present in this structured dataset and can't be added without new data sources.
- **Imperfect classifier calibration** — probability outputs are a ranking signal, not literal probabilities, without an added calibration step.
- **A specific, named classifier blind spot**: under-identifies exits reached via a lighter funding path (notebook 11).
- **`xgboost`/`shap`/`statsmodels` weren't set up while I was building notebooks 08-11**, so those sections either reimplement the method from scratch and validate it against my own published numbers, or reuse the SHAP plots I'd already generated with `src/ml/explainability.py`. Every place this happened is called out directly in the notebook itself, and nothing was faked to fill the gap.

## 6. Reproducibility statement

In [1]:
import platform
import sklearn, scipy, numpy, pandas, matplotlib, seaborn
print("Library versions used for this notebook series:")
print(f"  Python:      {platform.python_version()}")
print(f"  pandas:      {pandas.__version__}")
print(f"  numpy:       {numpy.__version__}")
print(f"  scikit-learn:{sklearn.__version__}")
print(f"  scipy:       {scipy.__version__}")
print(f"  matplotlib:  {matplotlib.__version__}")
print(f"  seaborn:     {seaborn.__version__}")
print("\nAll notebooks read data via RELATIVE paths (data/raw, data/warehouse, docs/) from the")
print("project root -- no absolute paths, no manual steps, no hidden state between cells within")
print("a notebook. Each notebook (09-11) re-derives any model it needs rather than depending on")
print("in-memory state from a previous notebook, so 'Run All' works standalone in every notebook.")

Library versions used for this notebook series:
  Python:      3.12.3
  pandas:      3.0.2
  numpy:       2.4.4
  scikit-learn:1.8.0
  scipy:       1.17.1
  matplotlib:  3.10.8
  seaborn:     0.13.2

All notebooks read data via RELATIVE paths (data/raw, data/warehouse, docs/) from the
project root -- no absolute paths, no manual steps, no hidden state between cells within
a notebook. Each notebook (09-11) re-derives any model it needs rather than depending on
in-memory state from a previous notebook, so 'Run All' works standalone in every notebook.


## 7. Future work

1. **Enrich features beyond Crunchbase's structured fields** — investor reputation/network centrality, founder track record, press/social sentiment — to close the ~50% unexplained-variance gap in the regression model.
2. **Recalibrate the classifier** (Platt scaling / isotonic regression) if probability outputs are ever used beyond ranking.
3. **Investigate the "lighter funding path" exit blind spot directly** — a dedicated sub-model or feature set for capital-efficient acquisition-track startups.
4. **Refresh with post-2015 data** — the single most important next step for making any dashboard claim about "current" startup trends actually current.
5. **Re-run the sections that used manual OLS/VIF/Breusch-Pagan reimplementations with `statsmodels`** directly installed, as a final cross-check beyond the coefficient-level validation already performed in notebook 08.

## 8. Final completeness check (against the original project brief)

In [1]:
import pandas as pd
completeness = pd.DataFrame([
    {"requirement": "Complete data profiling", "status": "Done", "notebook": "03"},
    {"requirement": "Complete data cleaning", "status": "Done", "notebook": "04"},
    {"requirement": "Complete EDA", "status": "Done", "notebook": "05"},
    {"requirement": "Complete statistical analysis", "status": "Done", "notebook": "08"},
    {"requirement": "Complete feature engineering", "status": "Done", "notebook": "06"},
    {"requirement": "Complete preprocessing", "status": "Done", "notebook": "07"},
    {"requirement": "Complete ML pipeline (regression, classification, clustering)", "status": "Done", "notebook": "09"},
    {"requirement": "Complete hyperparameter tuning", "status": "Done", "notebook": "09"},
    {"requirement": "Complete model evaluation", "status": "Done", "notebook": "10"},
    {"requirement": "Complete explainability", "status": "Done", "notebook": "11"},
    {"requirement": "Complete business insights", "status": "Done", "notebook": "12"},
    {"requirement": "Complete error analysis", "status": "Done", "notebook": "11"},
    {"requirement": "Complete reproducible notebooks", "status": "Done -- relative paths, self-contained", "notebook": "all"},
    {"requirement": "Complete documentation", "status": "Done -- markdown throughout + existing docs/ reports", "notebook": "all"},
    {"requirement": "Interview-ready explanations", "status": "Done -- every notebook", "notebook": "all"},
])
completeness

,requirement,status,notebook
0,Complete data profiling,Done,03
1,Complete data cleaning,Done,04
2,Complete EDA,Done,05
3,Complete statistical analysis,Done,08
4,Complete feature engineering,Done,06
5,Complete preprocessing,Done,07
6,"Complete ML pipeline (regression, classificati...",Done,09
7,Complete hyperparameter tuning,Done,09
8,Complete model evaluation,Done,10
9,Complete explainability,Done,11


## Closing note

This project was already a strong, working analytics platform before this notebook series existed. What this series adds isn't a redesign — it's a second, narrative layer that makes the *reasoning* behind the existing pipeline visible and checkable: every cleaning decision justified against a measured baseline, every statistical claim cross-validated against the project's own published numbers, every model comparison run live rather than asserted, and every limitation stated as plainly as the findings themselves.